# Introduzione

In questo progetto affronto il problema del domain shift tra immagini sintetiche e immagini reali nel riconoscimento di dispositivi di protezione individuale. Ho a disposizione un dataset sintetico che ho generato con Unity, completamente etichettato, e un insieme più piccolo di immagini reali. Un modello allenato solo sul sintetico funziona molto peggio sul reale, perché le due distribuzioni di immagini sono diverse (texture, luce, rumore). L’obiettivo è misurare questo divario e provare a ridurlo con una tecnica di domain adaptation avversaria (DANN), confrontandola con una baseline, con il transfer learning e con un modello “oracle” allenato direttamente sul reale.

### Organizzazione degli esperimenti
Nel primo tentativo il DANN collassava su una sola classe, esattamente come le baseline. Guardando l’addestramento ho visto che la parte avversaria diventava troppo forte (l’accuratezza del domain classifier scendeva sotto 0.5) e che, con pochi batch per epoca, le statistiche di BatchNorm della ResNet si adattavano completamente al sintetico. Per questo nella configurazione finale, usata in tutto il notebook, limito il peso massimo della parte avversaria a λ = 0.3 e tengo fisse le statistiche di BatchNorm della rete pre-allenata. Questa configurazione l’ho scelta in queste prove preliminari e poi non l’ho più modificata.

Ogni esperimento lo ripeto con 3 seed, perché con così pochi dati un singolo run può dipendere molto dal caso.

### 0. Configurazione
Tutti i parametri sono qui. La divisione delle immagini reali tra training e test usa un seed fisso SPLIT_SEED, così resta la stessa per tutti gli esperimenti. I seed in SEEDS cambiano solo l’inizializzazione e l’ordine dei batch.

In [ ]:
DATA_ROOT = "dataset_labeled"
EPOCHS = 30
BATCH_SIZE = 32
LR = 1e-3
WORKERS = 0               # su Windows nei notebook è più sicuro 0
LAMBDA_MAX = 0.3          # peso massimo della parte avversaria del DANN
SPLIT_SEED = 42           # seed fisso per la divisione del reale in train/test
SEEDS = [42, 0, 1]        # seed con cui ripeto ogni esperimento
SEED = SEEDS[0]

### 1. Import e trasformazioni
Definisco due pipeline di trasformazioni. In training applico data augmentation (crop casuale, flip orizzontale e variazione di colore), così il modello vede ogni immagine in versioni leggermente diverse e generalizza meglio. In test uso invece trasformazioni fisse (resize e crop centrale), perché la valutazione deve essere deterministica. Normalizzo con media e deviazione standard di ImageNet perché la rete che uso è pre-allenata su ImageNet.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightning as L
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.utilities import CombinedLoader
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from IPython.display import display
import warnings, logging

warnings.filterwarnings("ignore")
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

### 2. Dataset
Carico i due domini con ImageFolder, dove ogni sottocartella corrisponde a una classe, e controllo che le classi siano le stesse nei due domini. Divido le immagini reali in due metà in modo stratificato, dove la prima la uso durante il training del DANN ma senza usare le etichette, come succederebbe in un caso reale in cui le immagini vere non sono annotate, la seconda la tengo da parte solo per il test. La tabella mostra quante immagini ho per classe in ogni parte.

In [ ]:
src_train = datasets.ImageFolder(f"{DATA_ROOT}/synthetic", transform=train_tf)
src_eval = datasets.ImageFolder(f"{DATA_ROOT}/synthetic", transform=test_tf)
real_train = datasets.ImageFolder(f"{DATA_ROOT}/real", transform=train_tf)
real_eval = datasets.ImageFolder(f"{DATA_ROOT}/real", transform=test_tf)
assert src_train.classes == real_train.classes, "Le classi dei due domini devono coincidere"
CLASSES = src_train.classes

tr_idx, te_idx = train_test_split(np.arange(len(real_train)), test_size=0.5,
                                  stratify=real_train.targets, random_state=SPLIT_SEED)
tgt_train = Subset(real_train, tr_idx)
tgt_test = Subset(real_eval, te_idx)

def counts(targets):
    return pd.Series(np.array(targets)).map(dict(enumerate(CLASSES))).value_counts()

pd.DataFrame({
    "sintetico": counts(src_train.targets),
    "reale (train, no label)": counts(np.array(real_train.targets)[tr_idx]),
    "reale (test)": counts(np.array(real_train.targets)[te_idx]),
}).fillna(0).astype(int)

Esempi dei due domini in cui mostro alcune immagini sintetiche e alcune reali per vedere a occhio le differenze tra i due domini, ovvero quello che crea il domain shift.

In [ ]:
def show_examples(root, n=6):
    ds = datasets.ImageFolder(root)
    idx = np.random.RandomState(0).choice(len(ds), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.4))
    for ax, i in zip(axes, idx):
        img, y = ds[i]
        ax.imshow(img); ax.set_title(ds.classes[y], fontsize=9); ax.axis("off")
    plt.suptitle(root.split("/")[-1]); plt.show()

show_examples(f"{DATA_ROOT}/synthetic")
show_examples(f"{DATA_ROOT}/real")

### 3. Modello

Il modello è composto da tre parti. Il feature extractor è una ResNet18 a cui tolgo l’ultimo layer, così produce un vettore di feature per ogni immagine. Il label classifier usa queste feature per predire la classe ed è allenato solo sulle immagini sintetiche etichettate. Il domain classifier usa le stesse feature per capire se l’immagine è sintetica o reale. Tra feature extractor e domain classifier inserisco il gradient reversal layer: in avanti lascia passare le feature invariate, all’indietro inverte il segno del gradiente. In questo modo il domain classifier impara a distinguere i domini, mentre il feature extractor viene spinto a produrre feature che non permettono di distinguerli. Il peso λ di questa parte avversaria cresce gradualmente da 0 a un massimo di 0.3 durante il training, seguendo lo schedule di Ganin & Lempitsky (2015). All’inizio il modello impara prima il task e non viene destabilizzato. Infine, quando la rete è pre-allenata, tengo fisse le statistiche di BatchNorm di ImageNet, perché con così pochi batch per epoca si adatterebbero completamente al sintetico.

In [ ]:
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


class DANN(L.LightningModule):
    def __init__(self, num_classes, mode="dann", pretrained=True, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.feature_extractor = backbone
        self.label_classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(feat_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, num_classes))
        self.domain_classifier = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1))
        self.task_loss = nn.CrossEntropyLoss()
        self.domain_loss = nn.BCEWithLogitsLoss()
        self.preds, self.labels = [], []

    def forward(self, x):
        return self.label_classifier(self.feature_extractor(x))

    def grl_lambda(self):
        p = self.global_step / max(1, self.trainer.estimated_stepping_batches)
        return LAMBDA_MAX * (2.0 / (1.0 + math.exp(-10 * p)) - 1.0)

    def on_train_epoch_start(self):
        if self.hparams.pretrained:       # con backbone pre-allenata tengo le statistiche BatchNorm di ImageNet
            for m in self.feature_extractor.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eval()

    def training_step(self, batch, batch_idx):
        xs, ys = batch["src"]
        f_s = self.feature_extractor(xs)
        loss_task = self.task_loss(self.label_classifier(f_s), ys)
        loss = loss_task
        if self.hparams.mode == "dann":
            xt, _ = batch["tgt"]          # etichette del target NON usate
            f_t = self.feature_extractor(xt)
            lam = self.grl_lambda()
            feats = torch.cat([f_s, f_t])
            dom = torch.cat([torch.zeros(len(f_s)), torch.ones(len(f_t))]).to(self.device)
            dom_logits = self.domain_classifier(GradReverse.apply(feats, lam)).squeeze(1)
            loss_dom = self.domain_loss(dom_logits, dom)
            loss = loss_task + loss_dom
            dom_acc = ((dom_logits > 0).float() == dom).float().mean()
            self.log_dict({"train/loss_domain": loss_dom, "train/lambda": lam,
                           "train/domain_acc": dom_acc})
        self.log_dict({"train/loss_task": loss_task, "train/loss": loss})
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        self.preds.append(self(x).argmax(1).cpu())
        self.labels.append(y.cpu())

    def on_validation_epoch_end(self):
        p = torch.cat(self.preds).numpy(); y = torch.cat(self.labels).numpy()
        self.log("val/acc_target", accuracy_score(y, p))
        self.log("val/f1_target", f1_score(y, p, average="macro"))
        self.preds.clear(); self.labels.clear()

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=self.hparams.lr, momentum=0.9, weight_decay=5e-4)

### 4. Training e valutazione
Scrivo una funzione che allena un modello in una certa modalità e poi lo valuta. Le modalità sono:
- `source_only`: allenamento solo sulle immagini sintetiche (baseline e transfer learning);
- `dann`: immagini sintetiche con le etichette più immagini reali senza etichette;
- `finetune`: prima alleno sul sintetico, poi continuo l’allenamento sulle immagini reali etichettate;
- `oracle`: allenamento direttamente sulle immagini reali etichettate.

Per ogni esperimento salvo l’accuracy sul sintetico e sul reale, l’F1 macro (utile se le classi non sono bilanciate), la matrice di confusione e le feature estratte, che mi servono per il t-SNE. Uso SGD con momentum, come nei laboratori. Il modello che valuto è sempre quello dell’ultima epoca, l’accuracy sul test reale la registro su TensorBoard solo per vedere le curve, non per scegliere l’epoca migliore.

In [ ]:
@torch.no_grad()
def extract(model, loader):
    model.eval().to(device)
    feats, preds, labels = [], [], []
    for x, y in loader:
        f = model.feature_extractor(x.to(device))
        feats.append(f.cpu()); preds.append(model.label_classifier(f).argmax(1).cpu()); labels.append(y)
    return torch.cat(feats).numpy(), torch.cat(preds).numpy(), torch.cat(labels).numpy()


RESULTS = {}

def run_experiment(mode, pretrained=True):
    # mode: "source_only", "dann", "finetune" oppure "oracle"
    name = f"{mode}_{'pretrained' if pretrained else 'scratch'}"
    L.seed_everything(SEED)
    test_loader = DataLoader(tgt_test, BATCH_SIZE, num_workers=WORKERS)

    def train_loader(ds):
        return DataLoader(ds, BATCH_SIZE, shuffle=True, num_workers=WORKERS, drop_last=True)

    def fit(model, loaders, log_name):
        trainer = L.Trainer(max_epochs=EPOCHS, accelerator="auto", log_every_n_steps=1,
                            logger=TensorBoardLogger("tb_logs", name=log_name), enable_model_summary=False)
        trainer.fit(model, CombinedLoader(loaders, mode="max_size_cycle"), test_loader)

    if mode == "finetune":
        # fase 1: alleno sul sintetico, come nel transfer learning
        model = DANN(len(CLASSES), "source_only", pretrained, LR)
        fit(model, {"src": train_loader(src_train)}, f"{name}_seed{SEED}_fase1_sintetico")
        # fase 2: continuo l'allenamento sulle immagini reali etichettate
        fit(model, {"src": train_loader(tgt_train)}, f"{name}_seed{SEED}_fase2_reale")
    else:
        model = DANN(len(CLASSES), mode, pretrained, LR)
        loaders = {"src": train_loader(tgt_train if mode == "oracle" else src_train)}
        if mode == "dann":
            loaders["tgt"] = train_loader(tgt_train)      # immagini reali senza etichette
        fit(model, loaders, f"{name}_seed{SEED}")

    f_tgt, p, y = extract(model, test_loader)
    f_src, p_src, y_src = extract(model, DataLoader(src_eval, BATCH_SIZE, num_workers=WORKERS))
    RESULTS[name] = dict(acc=accuracy_score(y, p), f1=f1_score(y, p, average="macro"),
                         acc_src=accuracy_score(y_src, p_src),
                         cm=confusion_matrix(y, p, labels=range(len(CLASSES))),
                         f_src=f_src, f_tgt=f_tgt)
    print(f"{name}: acc sintetico = {RESULTS[name]['acc_src']:.3f} | "
          f"accuracy reale = {RESULTS[name]['acc']:.3f} | F1 macro = {RESULTS[name]['f1']:.3f}")
    return model

Queste due funzioni mostrano e salvano su disco (cartella `risultati`) tabelle e figure, così sono pronte per il report.

In [ ]:
SAVE_DIR = "risultati"
os.makedirs(SAVE_DIR, exist_ok=True)

def mostra_risultati(tag):
    """Mostra e salva tabella, matrici di confusione e t-SNE degli esperimenti in RESULTS."""
    # tabella
    table = pd.DataFrame({k: {"acc sintetico": v.get("acc_src", np.nan), "acc reale": v["acc"],
                              "F1 macro reale": v["f1"]} for k, v in RESULTS.items()}).T.round(3)
    display(table)
    table.to_csv(f"{SAVE_DIR}/{tag}_tabella.csv")

    # matrici di confusione
    fig, axes = plt.subplots(1, len(RESULTS), figsize=(4.5 * len(RESULTS), 4))
    for ax, (name, r) in zip(np.atleast_1d(axes), RESULTS.items()):
        ConfusionMatrixDisplay(r["cm"], display_labels=CLASSES).plot(ax=ax, colorbar=False)
        ax.set_title(name, fontsize=10)
    fig.suptitle(f"Matrici di confusione - {tag}")
    plt.tight_layout(); fig.savefig(f"{SAVE_DIR}/{tag}_confusione.png", dpi=150); plt.show()

    # t-SNE
    fig, axes = plt.subplots(1, len(RESULTS), figsize=(4.5 * len(RESULTS), 4))
    for ax, (name, r) in zip(np.atleast_1d(axes), RESULTS.items()):
        n_src = len(r["f_src"])
        X = np.concatenate([r["f_src"], r["f_tgt"]])
        emb = TSNE(2, random_state=0, perplexity=min(30, (len(X) - 1) // 3)).fit_transform(X)
        ax.scatter(*emb[:n_src].T, s=8, label="sintetico")
        ax.scatter(*emb[n_src:].T, s=8, label="reale")
        ax.set_title(name, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    np.atleast_1d(axes)[0].legend()
    fig.suptitle(f"t-SNE delle feature - {tag}")
    plt.tight_layout(); fig.savefig(f"{SAVE_DIR}/{tag}_tsne.png", dpi=150); plt.show()


def riassunto_seed(tag):
    """Mostra e salva i risultati seed per seed e la tabella media ± deviazione standard."""
    per_seed = runs_df.pivot(index="esperimento", columns="seed", values="acc_reale").round(3)
    summary = (runs_df.groupby("esperimento", sort=False)[["acc_sintetico", "acc_reale", "f1_reale"]]
               .agg(["mean", "std"]).round(3))
    display(per_seed); display(summary)
    runs_df.to_csv(f"{SAVE_DIR}/{tag}_tutti_i_run.csv", index=False)
    summary.to_csv(f"{SAVE_DIR}/{tag}_media_std.csv")

### 5. Esperimenti
Ripeto tutti gli esperimenti con tre seed (42, 0, 1). La divisione tra immagini reali di training e di test resta sempre la stessa, cambiano solo l’inizializzazione e l’ordine dei batch. Con un test reale di 41 immagini ogni immagine sposta l’accuracy di circa 2.4 punti, dunque un singolo run non basta.

- **Baseline da zero**: alleno la rete senza pre-allenamento usando solo le immagini sintetiche e la testo sulle reali; mi dice quanto è grande il domain gap.
- **Transfer learning**: ripeto lo stesso esperimento partendo dalla ResNet18 pre-allenata su ImageNet, per vedere quanto aiuta partire da feature già apprese su immagini reali, anche senza fare domain adaptation.
- **DANN**: alleno il modello con la domain adaptation avversaria, usando le immagini sintetiche con le etichette e le immagini reali senza etichette. Mi aspetto un miglioramento rispetto al transfer learning, perché il modello viene forzato a imparare feature che funzionano su entrambi i domini.
- **Fine-tuning**: parto dal modello allenato sul sintetico e lo rifinisco sulle immagini reali etichettate. Voglio capire se, avendo a disposizione poche immagini reali etichettate, partire dal sintetico aiuta.
- **Oracle**: alleno il modello direttamente sulle immagini reali etichettate, partendo solo da ImageNet. Serve come riferimento; confrontandolo con il fine-tuning vedo se il passaggio dal sintetico porta un vantaggio rispetto a usare solo le immagini reali.

DANN, baseline e transfer learning non usano nessuna etichetta del reale; fine-tuning e oracle invece sì, quindi non sono confrontabili alla pari con gli altri ma rispondono a una domanda diversa, ovvero quanto si guadagna se qualche immagine reale si etichetta.

In [ ]:
EXPERIMENTS = [("source_only", False), ("source_only", True), ("dann", True),
               ("finetune", True), ("oracle", True)]

runs = []
for s in SEEDS:
    SEED = s
    for mode, pre in EXPERIMENTS:
        run_experiment(mode, pre)
        name = f"{mode}_{'pretrained' if pre else 'scratch'}"
        r = RESULTS[name]
        runs.append({"seed": s, "esperimento": name, "acc_sintetico": r["acc_src"],
                     "acc_reale": r["acc"], "f1_reale": r["f1"]})
runs_df = pd.DataFrame(runs)

### 6. Risultati
La prima tabella riporta l’accuracy sul reale seed per seed, la seconda media e deviazione standard sui tre seed. Se il DANN funziona, la sua accuracy sul reale deve stare tra quella del transfer learning e quella dell’oracle.

In [ ]:
riassunto_seed("3seed")

### 7. Matrici di confusione e t-SNE
Le figure si riferiscono all’ultimo run eseguito (seed 1). La tabella riepiloga i risultati di quel run, mentre le matrici di confusione mostrano quali classi vengono confuse e se un modello predice sempre la stessa classe.

Con il t-SNE proietto in due dimensioni le feature estratte dalle immagini sintetiche e reali. Nella baseline mi aspetto due gruppi separati, segno che la rete rappresenta i due domini in modo diverso. Nel DANN invece i punti dovrebbero risultare più vicini o mescolati, il che vuol dire che il modello ha imparato feature più indipendenti dal dominio.

In [ ]:
mostra_risultati(f"ultimo_seed{SEEDS[-1]}")

### 8. Curve di training
Con TensorBoard controllo l’andamento delle loss durante il training. Per il DANN guardo soprattutto l’accuratezza del domain classifier: se scende verso 0.5, il classificatore non riesce più a distinguere sintetico da reale, cioè l’allineamento tra i due domini sta funzionando.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tb_logs --port 6008

Curve di training dei tre run DANN (seed 0, 1, 42):

![Loss del domain classifier](risultati/curve_dann_loss_domain.png)
![Accuratezza del domain classifier](risultati/curve_dann_domain_acc.png)
![Loss totale](risultati/curve_dann_loss.png)

Nelle prime ~20 iterazioni λ è quasi zero, quindi il domain classifier impara a distinguere i due domini: 
l'accuratezza sale fino a circa 0.9 e la loss_domain scende. Quando λ arriva al suo valore massimo (0.3) il feature extractor inizia a "ingannarlo" e l'accuratezza scende. Però non si ferma a 0.5 (che corrisponde a una loss_domain di circa ln 2 ≈ 0.69): continua a oscillare in modo diverso per ogni seed e nelle ultime epoche la loss_domain risale oltre 1. Questo mostra che il gioco avversario tra feature extractor e domain classifier non arriva a un equilibrio stabile.

### 9. Conclusioni

Riassumo i risultati sui 3 seed (accuracy sul test reale, 41 immagini, media ± deviazione standard):

| Esperimento | Etichette reali usate | Acc reale | F1 macro reale |
|---|---|---|---|
| Baseline CNN da zero (solo sintetico) | no | 0.528 ± 0.113 | 0.422 ± 0.182 |
| Transfer learning ResNet18 (solo sintetico) | no | 0.463 ± 0.000 | 0.317 ± 0.000 |
| DANN | no (solo immagini) | 0.683 ± 0.194 | 0.629 ± 0.273 |
| Fine-tuning sintetico → reale | sì (41) | 0.967 ± 0.014 | 0.967 ± 0.014 |
| Oracle (solo reale) | sì (41) | 0.959 ± 0.014 | 0.959 ± 0.014 |

**Domain gap.** Tutti i modelli allenati solo sul sintetico arrivano a circa il 100% sul sintetico e crollano sul reale. Il caso più evidente è il transfer learning dove, dalla matrice di confusione, si vede che classifica tutte le immagini reali come "senza giubbotto". Quindi quello che il modello ha imparato sul giubbotto renderizzato in Unity non si trasferisce al giubbotto vero. Anche il t-SNE lo conferma: sintetico e reale formano due cluster separati.

**DANN.** Senza usare nessuna etichetta del reale, il DANN è l'unico metodo che migliora rispetto alla baseline: in media +22 punti sul transfer learning, con 0.829 nel seed migliore. Però è molto instabile: in un seed resta al livello della baseline (0.463). Dai grafici di training si vede il perché. La domain_acc prima sale verso 0.9 (con λ piccolo il discriminatore distingue i domini), poi scende quando λ cresce, ma non si stabilizza a 0.5, continua a oscillare. Anche la loss_domain risale nelle ultime epoche. È il comportamento tipico del gioco minimax tra feature extractor e discriminatore, che non converge a un equilibrio. In più, con così pochi dati anche il non-determinismo della GPU sposta i risultati, infatti rilanciando con lo stesso seed ho ottenuto numeri diversi. Nel t-SNE i due domini restano comunque in gran parte separati, quindi l'allineamento è solo parziale.

**Fine-tuning e oracle.** Con le 41 immagini reali etichettate, sia il fine-tuning che l'oracle arrivano intorno al 96-97%. Il fine-tuning è circa uguale all'oracle, quindi in questo caso partire dal sintetico non porta un vantaggio rispetto a partire direttamente da ImageNet. Una differenza però c'è, ovvero che il modello fine-tunato mantiene di più il dominio sintetico (0.905 contro 0.704 dell'oracle), mentre l'oracle non l'ha mai visto.

**Cosa ne ricavo.** Il problema principale è il domain shift sintetico -> reale. Se non ho etichette sul reale, il DANN è l'unico dei metodi provati che riesce in parte a ridurlo, ma non in modo affidabile. Se posso etichettare anche poche decine di immagini reali, il fine-tuning risolve quasi del tutto il problema.

**Limiti.** Il test reale ha solo 41 immagini, quindi ogni immagine sbagliata vale circa 2.4 punti di accuracy, e ho usato solo 3 seed. Per questo le differenze piccole non sono significative, soprattutto quella tra fine-tuning e oracle. Lo split reale è fisso (`SPLIT_SEED = 42`), quindi tra un seed e l'altro cambiano solo l'inizializzazione e l'ordine dei batch.

### Ulteriori riferimenti
- Y. Ganin, V. Lempitsky, *Unsupervised Domain Adaptation by Backpropagation*, ICML 2015.
- K. He et al., *Deep Residual Learning for Image Recognition*, CVPR 2016.